# Round 1: CKA_S fine-tuning of CordsNet-R8 on monkey N

## Configurations

| | pretrained | control | student w0.1 | student w1 |
|---|---|---|---|---|
| start | original ImageNet checkpoint | pretrained | pretrained | pretrained |
| training | none (= epoch 0 of every run) | CE only | CE + 0.1 · L_CKA | CE + 1.0 · L_CKA |
| question | baseline | effect of fine-tuning on THINGS alone | weak alignment | strong alignment |

Shared settings (control and students):

| | |
|---|---|
| data split | monkey N, per concept 10 train / 1 val / 1 test images: 18,540 / 1,854 / 1,854 |
| batches | 64 images, all from one recording day; same batches for all runs (seed 2723) |
| epochs | 10 (288 steps each) |
| optimizer | AdamW, lr 3e-5, cosine to 0 per step, weight decay 0.01, fp32 |
| CE | pseudo-labels (pretrained model's prediction), stimulus steps 70-99 (142-200 ms), weights logspace(-3, 0, 30) |
| L_CKA | mean over bins of log(1 − CKA_S), IT vs layer8 |
| CKA loss bins | bins 7-19 (70-200 ms, IT response window), not all 20 bins |
| evaluation | val split, 27 batches of 64, all layers × areas × 20 bins, pooled over batches; after every epoch |
| response windows (analysis) | from the TVSD paper's window start (25 / 50 / 75 ms; bin containing it) to 200 ms: V1 20-200 ms (bins 2-19), V4 50-200 ms (bins 5-19), IT 70-200 ms (bins 7-19) |
| accuracy | val split, all 1,854 images, rerun from the checkpoints (`accuracy/eval_accuracy.py`) |

L_CKA in the control is computed without gradient (logged only).

In [ ]:
import warnings
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # nanmean over undefined bins

ROUND1 = Path("~/Desktop/DynVision/CordsNet/Round1").expanduser()
RUNS_DIR = ROUND1 / "outputs" / "round1"
NEURAL = Path("~/Desktop/neural_data").expanduser()         # raw monkey N npz files (Figure 5 only)

runs = ["control", "student_IT_layer8_w0p1", "student_IT_layer8_w1"]
label = {"pretrained": "pretrained", "control": "control",
         "student_IT_layer8_w0p1": "student w0.1", "student_IT_layer8_w1": "student w1"}
color = {"pretrained": "0.35", "control": "#2a78d6",
         "student_IT_layer8_w0p1": "#eb6834", "student_IT_layer8_w1": "#1baf7a"}

train = {run: pd.read_csv(RUNS_DIR / run / "train_metrics.csv") for run in runs}
val = {run: pd.read_csv(RUNS_DIR / run / "val_metrics.csv") for run in runs}
cfg = {run: yaml.safe_load(open(RUNS_DIR / run / "config_used.yaml")) for run in runs}
n_epochs = int(val[runs[0]]["epoch"].max())
ev = {run: [np.load(RUNS_DIR / run / "eval" / f"epoch_{e:03d}.npz") for e in range(n_epochs + 1)] for run in runs}

r0 = ev["control"][0]                                          # epoch 0 = pretrained (identical in all runs)
layers, areas, ests = [[str(x) for x in r0[k]] for k in ("layers", "areas", "estimators")]
S = ests.index("CKA_S")
t_ms = r0["bin_start_ms"] + 5                                  # bin centers
li, aj = layers.index("layer8"), areas.index("IT")
loss_bins = np.arange(cfg["student_IT_layer8_w1"]["loss"]["bins"][0], cfg["student_IT_layer8_w1"]["loss"]["bins"][1] + 1)
window = {"V1": np.arange(2, 20), "V4": np.arange(5, 20), "IT": np.arange(7, 20)}   # TVSD start (bin containing it) to 200 ms
steps_per_epoch = int(train["control"].groupby("epoch")["step"].max().iloc[0])
bin_cols = [f"cka_S_bin{k:02d}" for k in loss_bins]

# final models; "pretrained" = epoch 0
final = {"pretrained": r0, **{run: ev[run][n_epochs] for run in runs}}
order = ["pretrained"] + runs

# checks: config differences, epoch 0 identical across runs
print(pd.DataFrame({label[run]: {"w_cka": cfg[run]["loss"]["w_cka"], "CKA bins": cfg[run]["loss"]["bins"],
                                 "epochs": cfg[run]["train"]["epochs"], "batch": cfg[run]["train"]["batch_size"],
                                 "lr": cfg[run]["train"]["lr"]} for run in runs}))
print(f"{steps_per_epoch} steps per epoch, {n_epochs} epochs, {r0['h_num'].shape[0]} val batches")
print("epoch 0 max |diff| between runs:",
      max(np.nanmax(np.abs(ev[run][0]["cka_pooled"] - r0["cka_pooled"])) for run in runs))

## 1. Training curves

Per step, smoothed over 50 steps. Train CKA_S = IT vs layer8, per batch, mean over the loss bins.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for run in runs:
    d = train[run]
    x = d["global_step"] / steps_per_epoch
    axes[0].plot(x, d["loss_ce"].rolling(50, min_periods=1).mean(), color=color[run], lw=2, label=label[run])
    axes[1].plot(x, d[bin_cols].mean(axis=1).rolling(50, min_periods=1).mean(), color=color[run], lw=2, label=label[run])
axes[0].set(yscale="log", title="train CE (pseudo-labels)", ylabel="CE", xlabel="epoch")
axes[1].set(title="train CKA_S, IT vs layer8", ylabel="CKA_S", xlabel="epoch")
for ax in axes:
    ax.set_xticks(range(n_epochs + 1))
    ax.grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

## 2. Train vs val

Solid: train (mean over the epoch's steps). Dashed: val (after the epoch; epoch 0 = pretrained).
Train CKA_S is averaged over batches, val CKA_S is pooled over batches (H values averaged first), so they differ slightly by construction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for run in runs:
    tr = train[run].assign(cka=train[run][bin_cols].mean(axis=1)).groupby("epoch")[["loss_ce", "cka"]].mean()
    v = val[run]
    axes[0].plot(tr.index, tr["loss_ce"], "o-", color=color[run], lw=2, ms=4, label=label[run])
    axes[0].plot(v["epoch"], v["val_ce"], "o--", color=color[run], lw=2, ms=4)
    axes[1].plot(tr.index, tr["cka"], "o-", color=color[run], lw=2, ms=4)
    axes[1].plot(v["epoch"], v["CKA_S"], "o--", color=color[run], lw=2, ms=4)
axes[0].set(yscale="log", title="CE", ylabel="CE", xlabel="epoch")
axes[1].set(title=f"CKA_S, IT vs layer8, bins {loss_bins[0]}-{loss_bins[-1]}", ylabel="CKA_S", xlabel="epoch")
for ax in axes:
    ax.set_xticks(range(n_epochs + 1))
    ax.grid(alpha=0.3)
axes[0].plot([], [], "k-", label="train")
axes[0].plot([], [], "k--", label="val")
axes[0].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print(f"epoch {n_epochs}:")
for run in runs:
    tr = train[run][train[run]["epoch"] == n_epochs]
    v = val[run].set_index("epoch").loc[n_epochs]
    print(f"  {label[run]:13s} CE train {tr['loss_ce'].mean():.4f} val {v['val_ce']:.4f} | "
          f"CKA_S train {tr[bin_cols].mean(axis=1).mean():.3f} val {v['CKA_S']:.3f}")

## 3. When does alignment improve?

IT vs layer8, CKA_S per 10 ms bin (val). Gray: loss bins. ×: undefined bin.
Bottom: CKA_S per epoch and bin.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.axvspan(loss_bins[0] * 10, (loss_bins[-1] + 1) * 10, color="0.92", zorder=0)
ax.axhline(0, color="0.5", lw=1)
for run in order:
    y = final[run]["cka_pooled"][li, aj, :, S]
    ax.plot(t_ms, y, "o-", color=color[run], lw=2, ms=4, label=label[run])
    ax.plot(t_ms[np.isnan(y)], np.zeros(np.isnan(y).sum()), "x", color=color[run], ms=8, mew=2)
ax.set(title=f"IT vs layer8, epoch {n_epochs}", xlabel="time after image onset (ms)", ylabel="CKA_S", xlim=(0, 200))
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

heat = {run: np.array([ev[run][e]["cka_pooled"][li, aj, :, S] for e in range(n_epochs + 1)]) for run in runs}
vmax = np.nanmax([np.nanmax(h) for h in heat.values()])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, run in zip(axes, runs):
    im = ax.imshow(heat[run], aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=vmax,
                   extent=(0, 200, -0.5, n_epochs + 0.5))
    ax.axvline(loss_bins[0] * 10, color="w", lw=1, ls="--")
    ax.set(title=label[run], xlabel="time after image onset (ms)")
axes[0].set_ylabel("epoch")
fig.colorbar(im, ax=axes, label="CKA_S", shrink=0.9)
plt.show()

## 4. Other layers and areas

CKA_S averaged over each area's response window (V1 20-200 ms, V4 50-200 ms, IT 70-200 ms), epoch 10 vs pretrained.
Bottom: change from pretrained.

In [ ]:
win = {run: np.array([[np.nanmean(final[run]["cka_pooled"][i, j, window[a], S]) for j, a in enumerate(areas)]
                      for i in range(len(layers))]) for run in order}   # (layer, area)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for j, area in enumerate(areas):
    for run in order:
        axes[j].plot(range(1, 9), win[run][:, j], "o-", color=color[run], lw=2, ms=6, label=label[run])
    axes[j].set(title=area, xlabel="model layer", xticks=range(1, 9))
    axes[j].grid(alpha=0.3)
axes[0].set_ylabel("CKA_S (response window)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

delta = {run: win[run] - win["pretrained"] for run in runs}
lim = np.nanmax(np.abs(np.array(list(delta.values()))))
fig, axes = plt.subplots(1, 3, figsize=(16, 3.2), sharey=True)
for ax, run in zip(axes, runs):
    im = ax.imshow(delta[run].T, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
    for i in range(len(layers)):
        for j in range(len(areas)):
            ax.text(i, j, f"{delta[run][i, j]:+.2f}", ha="center", va="center", fontsize=8,
                    color="w" if abs(delta[run][i, j]) > 0.6 * lim else "k")
    ax.set(title=label[run], xticks=range(8), xticklabels=range(1, 9), xlabel="model layer",
           yticks=range(len(areas)), yticklabels=areas)
fig.colorbar(im, ax=axes, label="Δ CKA_S vs pretrained", shrink=0.9)
plt.show()

## 5. Model dynamics

Mean recorded activity of layer8 over time (val images). Left: scaled to first bin = 0, peak = 1, with monkey N IT (black, all trials and electrodes). Right: unscaled.

In [ ]:
it_total, it_n = np.zeros(20), 0
for f in sorted(NEURAL.rglob("monkeyN_day*_IT_binned.npz")):
    mua = np.load(f)["binned_mua"]
    it_total += mua.sum(axis=(1, 2))
    it_n += mua.shape[1] * mua.shape[2]
it_tc = it_total / it_n

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for run in order:
    c = final[run]["model_timecourse"][li]
    axes[0].plot(t_ms, (c - c[0]) / (c.max() - c[0]), "o-", color=color[run], lw=2, ms=4, label=label[run])
    axes[1].plot(t_ms, c, "o-", color=color[run], lw=2, ms=4, label=label[run])
if it_n > 0:
    axes[0].plot(t_ms, (it_tc - it_tc[0]) / (it_tc.max() - it_tc[0]), "o--", color="k", lw=2, ms=4, label="monkey N IT")
axes[0].set(title="layer8, normalized", ylabel="normalized activity")
axes[1].set(title="layer8, absolute", ylabel="mean activity")
for ax in axes:
    ax.set(xlabel="time after image onset (ms)", xlim=(0, 200))
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Time to half of the peak (bin center, ms), all layers:")
print(pd.DataFrame({label[run]: [t_ms[np.argmax((c - c[0]) / (c.max() - c[0]) >= 0.5)]
                                 for c in final[run]["model_timecourse"]] for run in order}, index=layers))

## 6. Val CE per time step

CE against the pseudo-labels at every stimulus step (val). Gray: CE loss steps 70-99.

In [ ]:
ce0, ce1 = cfg["control"]["loss"]["ce_steps"]
t_step = (np.arange(len(r0["ce_per_step"])) + 1) * 2          # ms after onset at the end of each 2 ms step
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax in axes:
    ax.axvspan(t_step[ce0] - 1, t_step[ce1 - 1] + 1, color="0.92", zorder=0)
    for run in order:
        ax.plot(t_step, final[run]["ce_per_step"], "-", color=color[run], lw=2, label=label[run])
    ax.set(xlabel="time after image onset (ms)", ylabel="val CE")
    ax.grid(alpha=0.3)
axes[0].set(title="all steps", xlim=(0, 200))
axes[1].set(title="CE loss steps", xlim=(t_step[ce0] - 4, 202),
            ylim=(0, 1.2 * max(final[run]["ce_per_step"][ce0:].max() for run in order)))
axes[0].legend()
plt.tight_layout()
plt.show()

## 7. Accuracy

Val split, all 1,854 images. Agreement: prediction = pseudo-label (pretrained model's prediction at 200 ms).
THINGS accuracy: only images of the 548 concepts that match ImageNet classes; correct if the prediction is one of the matched classes.
Top: at 200 ms, per epoch (epoch 0 = pretrained). Bottom: per time step, epoch 10; gray: CE loss steps.

In [ ]:
acc = {run: np.load(RUNS_DIR / run / "accuracy.npz") for run in runs}
a0 = acc["control"]
plabel, img_concept, t_acc, acc_epochs = a0["pseudo_label"], a0["concept"], a0["time_ms"], a0["epochs"]
overlap = {c: [int(i) for i in s.split(";")] for c, s in zip(a0["overlap_concepts"], a0["overlap_idx"])}
in_ov = np.isin(img_concept, list(overlap))
n_img = len(plabel)

# match[i, k] = True if ImageNet class k counts as correct for image i
match = np.zeros((n_img, 1000), dtype=bool)
for i, c in enumerate(img_concept):
    if c in overlap:
        match[i, overlap[c]] = True

agree, top1, top5 = {}, {}, {}                                   # each (epoch, time step)
for run in runs:
    t5 = acc[run]["top5"].astype(int)                            # (epoch, image, step, 5)
    hit = match[np.arange(n_img)[None, :, None, None], t5]
    agree[run] = (t5[..., 0] == plabel[None, :, None]).mean(axis=1)
    top1[run] = hit[..., 0][:, in_ov].mean(axis=1)
    top5[run] = hit[:, in_ov].any(axis=-1).mean(axis=1)

print(f"{n_img} val images, {in_ov.sum()} in ImageNet-overlap concepts")
print("epoch 0 identical across runs:", all(np.array_equal(acc[run]["top5"][0], a0["top5"][0]) for run in runs))
print(f"pretrained at 200 ms: agreement {agree['control'][0, -1]:.3f}, THINGS top-1 {top1['control'][0, -1]:.3f}, top-5 {top5['control'][0, -1]:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (name, d) in zip(axes, [("agreement with pseudo-labels", agree), ("THINGS top-1", top1), ("THINGS top-5", top5)]):
    ax.axhline(d["control"][0, -1], color=color["pretrained"], lw=1, ls="--", label="pretrained")
    for run in runs:
        ax.plot(acc_epochs, d[run][:, -1], "o-", color=color[run], lw=2, ms=4, label=label[run])
    ax.set(title=f"{name}, 200 ms", xlabel="epoch", xticks=acc_epochs)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("fraction of val images")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (name, d) in zip(axes, [("agreement with pseudo-labels", agree), ("THINGS top-1", top1)]):
    ax.axvspan(t_acc[ce0] - 1, t_acc[ce1 - 1] + 1, color="0.92", zorder=0)
    ax.plot(t_acc, d["control"][0], "-", color=color["pretrained"], lw=2, label="pretrained")
    for run in runs:
        ax.plot(t_acc, d[run][-1], "-", color=color[run], lw=2, label=label[run])
    ax.set(title=f"{name}, epoch {acc_epochs[-1]}", xlabel="time after image onset (ms)", xlim=(0, 200))
    ax.grid(alpha=0.3)
axes[0].set_ylabel("fraction of val images")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"at 200 ms, epoch {acc_epochs[-1]} (THINGS: n = {in_ov.sum()}, SE about 0.02):")
for name, run, e in [("pretrained", "control", 0)] + [(label[r], r, -1) for r in runs]:
    print(f"  {name:13s} agreement {agree[run][e, -1]:.3f}  top-1 {top1[run][e, -1]:.3f}  top-5 {top5[run][e, -1]:.3f}")